# ds004920 on Neurodesk: download one subject, visualize T1w and BOLD data with Niivue, and create FSL 3-column timing files

This example notebook is written in a **Neurodesk EDU** style for students who are beginning to work with open fMRI data in a BIDS dataset. The notebook does three things:

1. installs **ds004920** with **DataLad** and fetches the imaging files needed for one example subject,
2. visualizes a **T1w image** and **one BOLD run** with **Niivue** inside Jupyter, and
3. converts the dataset's available BIDS `_events.tsv` files into **FSL-style 3-column timing files** using the same **`bidsutils/BIDSto3col.sh`** approach used in lab.

The goal is to bridge the logic in introductory labs on MRI viewing and first-level modeling with a clean, reproducible notebook workflow that students can adapt for their own projects.

**Author:** David V. Smith  
**Date:** 2026-03-25  
**License:** MIT License

> **Authorship note:** This notebook was generated with ChatGPT, then revised extensively through instructor input, testing, and iteration. The instructor takes responsibility for the final content.

> Note: if this notebook uses tools provided by Neurodesk, please cite the respective tools and datasets in any downstream work.

### Citation and Resources

#### Tools included in this workflow
- **DataLad** for lightweight dataset installation and selective file retrieval: https://www.datalad.org/
- **FSL**: Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). *FSL*. *NeuroImage, 62*(2), 782–790. https://doi.org/10.1016/j.neuroimage.2011.09.015
- **Niivue / IPyNiiVue** for browser-based neuroimaging visualization in Jupyter: https://niivue.github.io/ipyniivue/
- **NiBabel** for working with NIfTI images in Python: https://nipy.org/nibabel/

#### Publications
- Gorgolewski, K. J., et al. (2016). *The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments*. *Scientific Data, 3*, 160044. https://doi.org/10.1038/sdata.2016.44

#### Educational resources
- This example is conceptually aligned with course materials on viewing MRI data and creating 3-column timing files for first-level modeling in FSL.
- Neurodesk example notebook template: https://neurodesk.org/edu/examples/template.html

#### Dataset
- **OpenNeuro ds004920, version 1.1.1**: *An fMRI dataset of social and nonsocial reward processing in young adults*. https://openneuro.org/datasets/ds004920/versions/1.1.1
- Smith, D. V., Wyngaarden, J., Sharp, C. J., Sazhin, D., Zaff, O., Fareri, D., & Jarcho, J. (2024). *An fMRI dataset of social and nonsocial reward processing in young adults*. *Data in Brief, 53*, 110197. https://doi.org/10.1016/j.dib.2024.110197

## Table of content
[1. Load software tools and import Python libraries](#1.-Load-software-tools-and-import-Python-libraries)  
[2. Parameters and helper functions](#2.-Parameters-and-helper-functions)  
[3. Install the dataset and fetch the imaging files we need](#3.-Install-the-dataset-and-fetch-the-imaging-files-we-need)  
[4. Summarize the dataset files available locally](#4.-Summarize-the-dataset-files-available-locally)  
[5. Visualize the T1w image and one BOLD run with Niivue](#5.-Visualize-the-T1w-image-and-one-BOLD-run-with-Niivue)  
[6. Create FSL 3-column timing files with bidsutils](#6.-Create-FSL-3-column-timing-files-with-bidsutils)  
[7. Inspect the generated timing files](#7.-Inspect-the-generated-timing-files)  
[8. Dependencies in Jupyter/Python](#8.-Dependencies-in-Jupyter/Python)

## 1. Load software tools and import Python libraries

In [ ]:
%%capture
!pip install -q ipyniivue nibabel pandas numpy matplotlib watermark

In [ ]:
import module
await module.load('fsl/6.0.7.16')
await module.list()

In [ ]:
from ipyniivue import NiiVue
from IPython.display import Markdown, display
from pathlib import Path
import json
import os
import re
import subprocess
import shlex

import nibabel as nib
import numpy as np
import pandas as pd

os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"

## 2. Parameters and helper functions

The notebook downloads a **single subject's imaging data** for visualization.

The BIDS `_events.tsv` files are plain-text files tracked directly in the dataset, so they should already be present after `datalad install` and `git checkout`. We therefore write derived timing files into `derivatives/` without modifying the raw BIDS data.

In [ ]:
DATASET_URL = "https://github.com/OpenNeuroDatasets/ds004920.git"
DATASET_TAG = "1.1.1"
DATASET_NAME = "ds004920"

SELECTED_SUBJECT = "sub-1001"
SELECTED_BOLD_BASENAME = f"{SELECTED_SUBJECT}_task-doors_run-1_bold.nii.gz"

HOME = Path.home()
DATASETS_HOME = HOME / "openneuro_datasets"
DATASET_ROOT = DATASETS_HOME / DATASET_NAME
DERIVATIVES_ROOT = DATASET_ROOT / "derivatives"
PREVIEW_ROOT = DERIVATIVES_ROOT / "notebook_preview"
THREECOL_ROOT = DERIVATIVES_ROOT / "fsl_3col"

BIDSUTILS_ROOT = HOME / "bidsutils"
BIDSTO3COL_SCRIPT = BIDSUTILS_ROOT / "BIDSto3col" / "BIDSto3col.sh"

def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def parse_bids_entities(path):
    name = Path(path).name.replace("_events.tsv", "")
    entities = {}
    for token in name.split("_"):
        if "-" in token:
            key, value = token.split("-", 1)
            entities[key] = value
    return entities

def subject_from_path(path):
    path = Path(path)
    for part in path.parts:
        if re.fullmatch(r"sub-[A-Za-z0-9]+", part):
            return part
    match = re.search(r"(sub-[A-Za-z0-9]+)", path.name)
    return match.group(1) if match else "unknown-sub"

def quote(pathlike):
    return shlex.quote(str(pathlike))

## 3. Install the dataset and fetch the imaging files we need

In [ ]:
DATASETS_HOME.mkdir(parents=True, exist_ok=True)

# Install the dataset into a dedicated datasets folder if it is not already present.
# We check for .git rather than DATASET_ROOT.exists() so a stray folder does not fool the notebook.
if not (DATASET_ROOT / ".git").exists():
    run(f"datalad install -s {DATASET_URL} {quote(DATASET_ROOT)}", cwd=HOME)

# Pin the dataset to the requested release tag.
run(f"git -c advice.detachedHead=false checkout {DATASET_TAG}", cwd=DATASET_ROOT)

# Create derivative folders only after the dataset exists.
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)
THREECOL_ROOT.mkdir(parents=True, exist_ok=True)

# Fetch the example files used for visualization.
run(f"datalad get {SELECTED_SUBJECT}/anat/{SELECTED_SUBJECT}_T1w.nii.gz", cwd=DATASET_ROOT)
run(f"datalad get {SELECTED_SUBJECT}/func/{SELECTED_BOLD_BASENAME}", cwd=DATASET_ROOT)

## 4. Summarize the dataset files available locally

This cell checks what is currently available after installation and checkout. The `_events.tsv` files should already be visible locally because they are tracked as text files in the repository rather than stored in the annex.

In [ ]:
t1_path = DATASET_ROOT / SELECTED_SUBJECT / "anat" / f"{SELECTED_SUBJECT}_T1w.nii.gz"
bold_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME
bold_json_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME.replace(".nii.gz", ".json")

events_files = sorted(DATASET_ROOT.glob("sub-*/func/*_events.tsv"))
if len(events_files) == 0:
    raise FileNotFoundError("No local _events.tsv files were found. The dataset install appears incomplete.")

event_entities = [parse_bids_entities(p) for p in events_files]
events_manifest = pd.DataFrame(event_entities).fillna("")

task_counts = (
    events_manifest.groupby("task")
    .size()
    .reset_index(name="n_event_files")
    .sort_values(["n_event_files", "task"], ascending=[False, True])
)

summary_text = f"""
**Selected subject for visualization:** `{SELECTED_SUBJECT}`  
**Selected BOLD run:** `{SELECTED_BOLD_BASENAME}`  
**Number of local `_events.tsv` files:** **{len(events_files)}**
"""
display(Markdown(summary_text))

display(task_counts)
events_manifest.head(10)

## 5. Visualize the T1w image and one BOLD run with Niivue

We first inspect the anatomical image and one functional run for `sub-1001`. The BOLD file is 4D, so we also create a representative **middle volume** for a lighter-weight overlay display.

In [ ]:
t1_img = nib.load(str(t1_path))
bold_img = nib.load(str(bold_path))

with open(bold_json_path) as f:
    bold_meta = json.load(f)

middle_vol_idx = bold_img.shape[3] // 2
middle_bold_data = np.asarray(bold_img.dataobj[..., middle_vol_idx], dtype=np.float32)
middle_bold_img = nib.Nifti1Image(middle_bold_data, bold_img.affine, bold_img.header)

middle_bold_path = PREVIEW_ROOT / SELECTED_BOLD_BASENAME.replace(".nii.gz", f"_middle-vol-{middle_vol_idx:03d}.nii.gz")
nib.save(middle_bold_img, str(middle_bold_path))

t1_zooms = t1_img.header.get_zooms()[:3]
bold_zooms = bold_img.header.get_zooms()[:4]

meta_text = f"""
### Quick metadata summary

- **T1w shape:** `{t1_img.shape}`
- **T1w voxel size (mm):** `{tuple(round(x, 3) for x in t1_zooms)}`
- **BOLD shape:** `{bold_img.shape}`
- **BOLD voxel size + TR:** `{tuple(round(x, 3) for x in bold_zooms)}`
- **BOLD RepetitionTime (s):** `{bold_meta.get("RepetitionTime", "n/a")}`
- **BOLD TaskName:** `{bold_meta.get("TaskName", "n/a")}`
- **Representative BOLD volume for overlays:** volume index `{middle_vol_idx}`
"""
display(Markdown(meta_text))

In [ ]:
# T1w anatomy
nv = NiiVue()
nv.load_volumes([
    {"path": str(t1_path), "name": f"{SELECTED_SUBJECT} T1w", "colormap": "gray"}
])
nv

In [ ]:
# Full 4D BOLD run
# In many Jupyter frontends, Niivue will expose a time/volume control for 4D images.
nv = NiiVue()
nv.load_volumes([
    {"path": str(bold_path), "name": SELECTED_BOLD_BASENAME, "colormap": "gray"}
])
nv

In [ ]:
# Optional overlay: T1w anatomy with a representative middle BOLD volume in red
nv = NiiVue()
nv.load_volumes([
    {"path": str(t1_path), "name": "T1w", "colormap": "gray"},
    {"path": str(middle_bold_path), "name": "BOLD middle volume", "colormap": "red", "opacity": 0.6},
])
nv

## 6. Create FSL 3-column timing files with bidsutils

In lab, we used the `BIDSto3col.sh` script from `bidsutils` like this:

```bash
cd ~
git clone https://github.com/bids-standard/bidsutils.git
cd bidsutils/BIDSto3col
bash BIDSto3col.sh ~/ds000157/sub-01/func/sub-01_task-passiveimageviewing_events.tsv sub-01
```

Below, we keep that same basic workflow but automate it across all local `_events.tsv` files in **ds004920**. The outputs are written to:

`derivatives/fsl_3col/sub-XXXX/func/`

That keeps the raw BIDS dataset unchanged while producing FEAT-ready timing files.

In [ ]:
if not BIDSUTILS_ROOT.exists():
    run(f"git clone https://github.com/bids-standard/bidsutils.git {quote(BIDSUTILS_ROOT)}", cwd=HOME)

if not BIDSTO3COL_SCRIPT.exists():
    raise FileNotFoundError(f"Cannot find {BIDSTO3COL_SCRIPT}")

threecol_records = []

for events_path in events_files:
    events_path = Path(events_path)
    subject = subject_from_path(events_path)
    entities = parse_bids_entities(events_path)
    task = entities.get("task", "")
    run_id = entities.get("run", "")
    entity_root = events_path.name.replace("_events.tsv", "")

    out_dir = THREECOL_ROOT / subject / "func"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Remove older outputs for this run so rerunning the notebook stays tidy.
    for old_file in out_dir.glob(f"{entity_root}*.txt"):
        old_file.unlink()

    run(
        f"bash {quote(BIDSTO3COL_SCRIPT)} {quote(events_path)} {quote(entity_root)}",
        cwd=out_dir,
    )

    generated_files = sorted(out_dir.glob(f"{entity_root}*.txt"))
    for out_file in generated_files:
        condition = out_file.stem.replace(entity_root, "").lstrip("_-.")
        threecol_records.append(
            {
                "subject": subject,
                "task": task,
                "run": run_id,
                "events_tsv": str(events_path.relative_to(DATASET_ROOT)),
                "condition": condition or "all",
                "three_col_file": str(out_file.relative_to(DATASET_ROOT)),
            }
        )

In [ ]:
threecol_manifest = pd.DataFrame(threecol_records).sort_values(
    ["subject", "task", "run", "condition", "three_col_file"]
).reset_index(drop=True)

manifest_path = THREECOL_ROOT / "manifest.tsv"
threecol_manifest.to_csv(manifest_path, sep="\t", index=False)

manifest_text = f"""
Created **{len(threecol_manifest)}** 3-column files from **{len(events_files)}** events files.  
A manifest was written to: `{manifest_path.relative_to(DATASET_ROOT)}`
"""
display(Markdown(manifest_text))

threecol_manifest.head(20)

## 7. Inspect the generated timing files

In [ ]:
summary_by_task = (
    threecol_manifest.groupby(["task", "condition"])
    .size()
    .reset_index(name="n_threecol_files")
    .sort_values(["task", "condition"])
)

display(summary_by_task)

example_candidates = threecol_manifest.query(
    "subject == @SELECTED_SUBJECT and task == 'doors' and run == '1'"
)

if len(example_candidates) == 0:
    example_row = threecol_manifest.iloc[0]
else:
    example_row = example_candidates.iloc[0]

example_threecol = DATASET_ROOT / example_row["three_col_file"]
display(Markdown(f"### Example output file\n`{example_row['three_col_file']}`"))

pd.read_csv(
    example_threecol,
    sep=r"\s+",
    header=None,
    names=["onset", "duration", "weight"]
).head(10)

### Notes for students

- These text files are ready to use as **custom 3-column EVs** in FSL FEAT.
- Because the outputs are written under `derivatives/`, the original BIDS dataset remains unchanged.
- The notebook mirrors the lab strategy of using **`BIDSto3col.sh`** rather than re-implementing the conversion logic from scratch in Python.
- For first-level modeling, you would still decide which conditions to keep separate and which to combine based on your hypothesis.

## 8. Dependencies in Jupyter/Python

In [ ]:
%load_ext watermark
%watermark -iv

print("\nCommand line tools:")
run("datalad --version")
run("git --version")